# Stop Delay Change

Compares delay changes by next stop, or by city part when a stop-to-city-part mapping is available.

In [1]:
from pathlib import Path
import importlib.util
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "analysis"))

spec = importlib.util.spec_from_file_location(
    "stop_delay_change",
    PROJECT_ROOT / "analysis" / "stop-delay-change.py",
)
stop_delay_change = importlib.util.module_from_spec(spec)
spec.loader.exec_module(stop_delay_change)

DB = PROJECT_ROOT / "data" / "foli.db"
TIMEZONE = "Europe/Helsinki"
GTFS_DIR = None
LIMIT = 20
MIN_OBSERVATIONS = 100
GROUP_BY = "stop"  # "stop" or "city-part"
CITY_PARTS_CSV = None  # Example: PROJECT_ROOT / "data" / "stop-city-parts.csv"
SORT_BY = "absolute"  # "absolute", "increase", or "decrease"
LINE_REF = None
BASELINE_START = None
BASELINE_END = None
COMPARISON_START = None
COMPARISON_END = None
DIRECTION_REF = None
QUALITY_MODE = "conservative"
BUCKET = "trip-stop"
LEGACY_MIDPOINT = True

By default this notebook uses the legacy midpoint split so it runs against the bundled database. For matched periods, set all four period variables before running the table cell. The comparison uses only matching line, direction, local weekday, and local hour contexts. To aggregate by city part, set `GROUP_BY = "city-part"` and point `CITY_PARTS_CSV` at a CSV with `stop_id,city_part` columns.

In [2]:
class Args:
    db = DB
    timezone = TIMEZONE
    gtfs_dir = GTFS_DIR
    city_parts_csv = CITY_PARTS_CSV
    group_by = GROUP_BY
    sort_by = SORT_BY
    line_ref = LINE_REF
    direction_ref = DIRECTION_REF
    baseline_start = BASELINE_START
    baseline_end = BASELINE_END
    comparison_start = COMPARISON_START
    comparison_end = COMPARISON_END
    min_observations = MIN_OBSERVATIONS
    limit = LIMIT
    quality_mode = QUALITY_MODE
    exclude_stop_call_disagreement = False
    bucket = BUCKET
    legacy_midpoint = LEGACY_MIDPOINT

observations = stop_delay_change.load_observations(Args)
change, period_description = stop_delay_change.build_stop_change(Args, observations)
print(period_description)
change

baseline=2026-04-23T09:45:00+00:00..2026-04-25T07:48:10+00:00, comparison=2026-04-25T07:48:10+00:00..2026-04-27T05:51:20.000001+00:00


,stop_id,baseline_bucket_count,baseline_raw_poll_count,baseline_signed_mean_delay_min,baseline_avg_delay_min,baseline_median_delay_min,baseline_p75_delay_min,baseline_p90_delay_min,baseline_p95_delay_min,baseline_pct_late,...,comparison_p90_delay_min,comparison_p95_delay_min,comparison_pct_late,comparison_pct_over_3_min_late,comparison_pct_over_5_min_late,comparison_pct_early,comparison_pct_over_1_min_early,comparison_pct_over_3_min_early,comparison_median_early_min_abs,comparison_p90_early_min_abs


In [3]:
if not change.empty:
    label_column = "city_part" if GROUP_BY == "city-part" else "stop_name"
    ax = change.sort_values("p90_delay_change_min").plot.barh(
        x=label_column,
        y="p90_delay_change_min",
        legend=False,
        figsize=(10, 6),
        title="Delay change by stop or city part",
    )
    ax.set_xlabel("Comparison minus baseline P90 delay, minutes")
    ax.set_ylabel("City part" if GROUP_BY == "city-part" else "Stop")